# CIFAR 用の DeepInversion+ ノートブック
交差エントロピー損失ではなく，プロトタイプと再構成画像の特徴の類似度を条件付けとする．

In [1]:
import os
import sys
import glob
import numpy as np
import json
import random
import collections


import torch
import torch.optim as optim
import torchvision.utils as vutils

import torch.nn as nn
import torch.nn.functional as F



## GPUの設定

In [2]:
# 使用するgpuを指定
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

## パスの設定

In [3]:
# ベース部分のパス
ckpt_path = "/home/kouyou/ContinualLearning/repexp/NeurIPS2024-PRL/checkpoint"

# cifar100のbaseline用パス
base_cifar100_path = "baseline/cifar100"

# baseline
method = "baseline"
baseline_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0_0_0")

# baseline_mu
method = "baseline_mu"
baseline_mu_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0_0_1.0")
# print("baseline_mu_path: ", baseline_mu_path)

# prl2
method = "prl2"
prl_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0.1_2_1.0")
# print("prl_path: ", prl_path)

# prl_mu
method = "prl-mu"
prl_mu_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0.1_2_1.0")
# print("prl_mu_path: ", prl_mu_path)

## Jsonファイルの読み込みからモデルの用意

In [9]:
# 手法名（変更箇所）
method = "BASELINE"

# プロジェクト root を sys.path に追加
project_root = "/home/kouyou/ContinualLearning/repexp/NeurIPS2024-PRL"
sys.path.append(project_root)

from utils import factory
import models

# --- 1) 設定を記述した jsonファイル の内容を読む ---
with open(os.path.join(project_root, "exps", method, "cifar.json")) as f:
    args = json.load(f)


args["device"] = ["0"]            # 必要に応じて
# args["model_name"] = "baseline" 


# --- 2) learner とネットワークの作成 ---
learner = factory.get_model(args["model_name"], args)
net = learner._network


# --- 3) checkpoint の読み込み ---
ckpt_dir = baseline_path
ckpt_file = os.path.join(ckpt_dir, "phase0.pkl")       # 読み込むモデルの指定

ckpt = torch.load(ckpt_file, map_location="cuda:0")
state_dict = ckpt["model_state_dict"]
print(state_dict.keys())

# fc の出力次元を checkpoint から取得
num_outputs = state_dict["fc.weight"].shape[0]
print("num_outputs: ", num_outputs)

# fc層の出力次元数を変更
net.update_fc(num_outputs)

# state_dict の読み込み
net.load_state_dict(state_dict)

# protos, forget_classes も保存されていれば復元
if "protos" in ckpt:
    net._protos = ckpt["protos"]
if "forget_classes" in ckpt and hasattr(net, "forget_classes"):
    net.forget_classes = ckpt["forget_classes"]

net.cuda().eval()

# 忘却クラスや class_order を取り出す
forget_classes = ckpt.get("forget_classes", None)
class_order = ckpt.get

odict_keys(['convnet.conv1.0.weight', 'convnet.conv1.1.weight', 'convnet.conv1.1.bias', 'convnet.conv1.1.running_mean', 'convnet.conv1.1.running_var', 'convnet.conv1.1.num_batches_tracked', 'convnet.layer1.0.conv1.weight', 'convnet.layer1.0.bn1.weight', 'convnet.layer1.0.bn1.bias', 'convnet.layer1.0.bn1.running_mean', 'convnet.layer1.0.bn1.running_var', 'convnet.layer1.0.bn1.num_batches_tracked', 'convnet.layer1.0.conv2.weight', 'convnet.layer1.0.bn2.weight', 'convnet.layer1.0.bn2.bias', 'convnet.layer1.0.bn2.running_mean', 'convnet.layer1.0.bn2.running_var', 'convnet.layer1.0.bn2.num_batches_tracked', 'convnet.layer1.1.conv1.weight', 'convnet.layer1.1.bn1.weight', 'convnet.layer1.1.bn1.bias', 'convnet.layer1.1.bn1.running_mean', 'convnet.layer1.1.bn1.running_var', 'convnet.layer1.1.bn1.num_batches_tracked', 'convnet.layer1.1.conv2.weight', 'convnet.layer1.1.bn2.weight', 'convnet.layer1.1.bn2.bias', 'convnet.layer1.1.bn2.running_mean', 'convnet.layer1.1.bn2.running_var', 'convnet.layer

<ipython-input-9-5b6dcb354779>:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_file, map_location="cuda:0")


## Hookの設定

In [5]:
class DeepInversionFeatureHook():
    '''
    Implementation of the forward hook to track feature statistics and compute a loss on them.
    Will compute mean and variance, and will use l2 as a loss
    '''

    def __init__(self, module):
        self.hook = module.register_forward_hook(self.hook_fn)


    def hook_fn(self, module, input, output):
        # hook co compute deepinversion's feature distribution regularization
        nch = input[0].shape[1]

        mean = input[0].mean([0, 2, 3])
        var = input[0].permute(1, 0, 2, 3).contiguous().view([nch, -1]).var(1, unbiased=False)

        # forcing mean and variance to match between two distributions
        # other ways might work better, e.g. KL divergence
        r_feature = torch.norm(module.running_var.data.type(var.type()) - var, 2) + torch.norm(
            module.running_mean.data.type(var.type()) - mean, 2)

        self.r_feature = r_feature
        # must have no output

    def close(self):
        self.hook.remove()


In [30]:
# 設定
args = {}

# ベスト更新
# args["bs"] = 100
# args["iters_mi"] = 5000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.05
# args["di_ce_scale"] = 1.7
# args["di_var_scale"] = 0
# args["di_l2_scale"] = 3.5e-5
# args["r_feature_weight"] = 0.30
# args["exp_descr"] = "debug_cifar100_v21"
# args["size"] = 32


# # 21からエポック数を増加
# args["bs"] = 100
# args["iters_mi"] = 10000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.05
# args["di_ce_scale"] = 1.7
# args["di_var_scale"] = 0
# args["di_l2_scale"] = 3.5e-5
# args["r_feature_weight"] = 0.30
# args["exp_descr"] = "debug_cifar100_v22"
# args["size"] = 32


# 22からceの重みを少し増加
# args["bs"] = 100
# args["iters_mi"] = 10000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.05
# args["di_ce_scale"] = 1.8
# args["di_var_scale"] = 0
# args["di_l2_scale"] = 3.5e-5
# args["r_feature_weight"] = 0.30
# args["exp_descr"] = "debug_cifar100_v23"
# args["size"] = 32


# # 23からceの重みを少し増加
# args["bs"] = 100
# args["iters_mi"] = 10000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.05
# args["di_ce_scale"] = 1.9
# args["di_var_scale"] = 0
# args["di_l2_scale"] = 3.5e-5
# args["r_feature_weight"] = 0.30
# args["exp_descr"] = "debug_cifar100_v24"
# args["size"] = 32


# # 24からceの重みを少し増加
# args["bs"] = 100
# args["iters_mi"] = 10000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.05
# args["di_ce_scale"] = 2.0
# args["di_var_scale"] = 0
# args["di_l2_scale"] = 3.5e-5
# args["r_feature_weight"] = 0.30
# args["exp_descr"] = "debug_cifar100_v25"
# args["size"] = 32


# # 25からr_featの重みを少し増加
# args["bs"] = 100
# args["iters_mi"] = 10000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.05
# args["di_ce_scale"] = 2.0
# args["di_var_scale"] = 0
# args["di_l2_scale"] = 3.5e-5
# args["r_feature_weight"] = 0.31
# args["exp_descr"] = "debug_cifar100_v26"
# args["size"] = 32


# # 26からr_featの重みを少し増加
# args["bs"] = 100
# args["iters_mi"] = 10000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.05
# args["di_ce_scale"] = 2.0
# args["di_var_scale"] = 0
# args["di_l2_scale"] = 3.5e-5
# args["r_feature_weight"] = 0.32
# args["exp_descr"] = "debug_cifar100_v27"
# args["size"] = 32


# # 27からエポック数を増加
# args["bs"] = 100
# args["iters_mi"] = 15000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.05
# args["di_ce_scale"] = 2.0
# args["di_var_scale"] = 0
# args["di_l2_scale"] = 3.5e-5
# args["r_feature_weight"] = 0.32
# args["exp_descr"] = "debug_cifar100_v28"
# args["size"] = 32


# # 27からエポック数を増加
# args["bs"] = 100
# args["iters_mi"] = 20000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.05
# args["di_ce_scale"] = 2.0
# args["di_var_scale"] = 0
# args["di_l2_scale"] = 3.5e-5
# args["r_feature_weight"] = 0.32
# args["exp_descr"] = "debug_cifar100_v29"
# args["size"] = 32


# # 21からfeature_weightを少し増加
# args["bs"] = 100
# args["iters_mi"] = 5000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.05
# args["di_ce_scale"] = 1.7
# args["di_var_scale"] = 0
# args["di_l2_scale"] = 3.5e-5
# args["r_feature_weight"] = 0.33
# args["exp_descr"] = "debug_cifar100_v30"
# args["size"] = 32


# # 30からl2_scaleを少し減少
# args["bs"] = 100
# args["iters_mi"] = 5000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.05
# args["di_ce_scale"] = 1.7
# args["di_var_scale"] = 0
# args["di_l2_scale"] = 3.3e-5
# args["r_feature_weight"] = 0.33
# args["exp_descr"] = "debug_cifar100_v31"
# args["size"] = 32


# # # 31からl2_scaleを少し減少
# args["bs"] = 100
# args["iters_mi"] = 5000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.05
# args["di_ce_scale"] = 1.7
# args["di_var_scale"] = 0
# args["di_l2_scale"] = 2.5e-5
# args["r_feature_weight"] = 0.33
# args["exp_descr"] = "debug_cifar100_v32"
# args["size"] = 32


# 32からl2_scaleを0に
args["bs"] = 100
args["iters_mi"] = 5000
args["cig_scale"] = 0.0
args["di_lr"] = 0.05
args["di_ce_scale"] = 1.7
args["di_var_scale"] = 0
args["di_l2_scale"] = 0
args["r_feature_weight"] = 0.33
args["exp_descr"] = "debug_cifar100_v33"
args["size"] = 32


random_labels = False

torch.manual_seed(777)
random.seed(777)

# 損失関数
criterion = nn.CrossEntropyLoss()

# 入力
data_type = torch.float
inputs = torch.randn((args["bs"], 3, args["size"], args["size"]), requires_grad=True, device='cuda', dtype=data_type)

# 最適化手法
optimizer = optim.Adam([inputs], lr=args["di_lr"])

batch_idx = 0
prefix = "runs/data_generation_di+/"+args["exp_descr"]+"/"

for create_folder in [prefix, prefix+"/best_images/"]:
    if not os.path.exists(create_folder):
        os.makedirs(create_folder)

global_iteration = 0

In [31]:
best_cost = 1e6

# 入力の初期化
inputs.data = torch.randn((args["bs"], 3, args["size"], args["size"]), requires_grad=True, device='cuda')

# Optimizer の初期化
optimizer.state = collections.defaultdict(dict)

# ラベルの作成
if random_labels:
    targets = torch.LongTensor([random.randint(0,9) for _ in range(args["bs"])]).to('cuda')
else:
    targets = torch.LongTensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9,
                                10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
                                20, 21, 22, 23, 24, 25, 26, 27, 28, 29,
                                30, 31, 32, 33, 34, 35, 36, 37, 38, 39,
                                40, 41, 42, 43, 44, 45, 46, 47, 48, 49] * 2).to('cuda')


# ============================================
# プロトタイプを用意（通常のDeepInversionと異なる箇所）
# ============================================
protos_dict = net._protos  # {label: proto_vec}
proto_labels = sorted(protos_dict.keys())
# print(protos_dict.keys())
# print(proto_labels)

# 各 proto を tensor にして並べる
protos_list = []
for c in proto_labels:
    v = protos_dict[c]                         # np.array or torch.Tensor
    v = torch.as_tensor(v, dtype=torch.float32)
    protos_list.append(v)

protos_tensor = torch.stack(protos_list, dim=0).to("cuda")  # (C, D)
protos_tensor = F.normalize(protos_tensor, dim=1)           # 行方向 L2 正規化

# ラベル → 行index のマップを作っておく
label2row = {c: i for i, c in enumerate(proto_labels)}
C, D = protos_tensor.shape
print("num protos:", C, "feat dim:", D)

## Create hooks for feature statistics catching
loss_r_feature_layers = []
for module in net.modules():
    if isinstance(module, nn.BatchNorm2d):
        loss_r_feature_layers.append(DeepInversionFeatureHook(module))

# setting up the range for jitter
lim_0, lim_1 = 2, 2


# 学習部分
for epoch in range(args["iters_mi"]):
    
    # apply random jitter offsets
    off1 = random.randint(-lim_0, lim_0)
    off2 = random.randint(-lim_1, lim_1)
    inputs_jit = torch.roll(inputs, shifts=(off1,off2), dims=(2,3))

    # foward with jit images
    optimizer.zero_grad()
    net.zero_grad()
    outputs = net(inputs_jit)

    # # 交差エントロピー損失
    # logits_all = outputs["logits"]
    # logits = logits_all[:, ::4] 
    # loss = criterion(logits, targets)
    # loss_target = loss.item()

    # print(outputs.keys())   # dict_keys(['logits', 'features'])

    # 特徴量ベースのターゲット損失
    feature = outputs["features"]
    feat = feature.view(feature.size(0), -1)                # 念のため flatten
    feat = F.normalize(feat, dim=1)                         # (bs, D), L2-normalize

    # protos_tensor: (C, D) をループ外で作っておいたものを使う
    sim = torch.matmul(feat, protos_tensor.t())             # (bs, C)

    # targets は「net._protos の key」と同じラベル空間と仮定
    # label2row で「ラベル → protos_tensor の行 index」に変換する
    row_idx = torch.tensor(
        [label2row[int(t.item())] for t in targets],
        device=feat.device,
        dtype=torch.long
    )           

    # 自分のクラスのプロトタイプとの類似度だけ抜き出す
    sim_pos = sim[torch.arange(feat.size(0), device=feat.device), row_idx]

    # 類似度を最大化したいので、マイナスを取って loss にする
    loss = - sim_pos.mean()
    loss_target = loss.item()

    # apply total variation regularization
    diff1 = inputs_jit[:,:,:,:-1] - inputs_jit[:,:,:,1:]
    diff2 = inputs_jit[:,:,:-1,:] - inputs_jit[:,:,1:,:]
    diff3 = inputs_jit[:,:,1:,:-1] - inputs_jit[:,:,:-1,1:]
    diff4 = inputs_jit[:,:,:-1,:-1] - inputs_jit[:,:,1:,1:]
    loss_var = torch.norm(diff1) + torch.norm(diff2) + torch.norm(diff3) + torch.norm(diff4)
    loss = args["di_ce_scale"] * loss + args["di_var_scale"] * loss_var

    # R_feature loss
    loss_distr = sum([mod.r_feature for mod in loss_r_feature_layers])
    loss = loss + args["r_feature_weight"] * loss_distr # best for noise before BN

    # l2 loss
    loss_l2 = torch.norm(inputs_jit, 2)
    loss = loss + args["di_l2_scale"] * loss_l2 


    if epoch % 10==0:
        print(f"It {epoch}\t Losses: total: {loss.item():3.3f},\ttarget: {loss_target:3.3f} \tR_feature_loss unscaled:\t {loss_distr.item():3.3f} \tl2 loss unscaled: {loss_l2.item():3.3f}")
        vutils.save_image(inputs.data.clone(),
                            './{}/output_{}.png'.format(prefix, epoch//10),
                            normalize=True, scale_each=True, nrow=10)

    if best_cost > loss.item():
        best_cost = loss.item()
        best_inputs = inputs.data
    

    # 最適化実行
    loss.backward()
    optimizer.step()



name_use = "best_images"
if prefix is not None:
    name_use = prefix + name_use
next_batch = len(glob.glob("./%s/*.png" % name_use)) // 1

vutils.save_image(best_inputs[:20].clone(),
                    './{}/output_{}.png'.format(name_use, next_batch),
                    normalize=True, scale_each = True, nrow=10)

num protos: 50 feat dim: 512
It 0	 Losses: total: 77.298,	target: -0.395 	R_feature_loss unscaled:	 236.273 	l2 loss unscaled: 554.592
It 10	 Losses: total: 42.915,	target: -0.560 	R_feature_loss unscaled:	 132.927 	l2 loss unscaled: 517.213
It 20	 Losses: total: 30.146,	target: -0.599 	R_feature_loss unscaled:	 94.436 	l2 loss unscaled: 492.104
It 30	 Losses: total: 22.748,	target: -0.609 	R_feature_loss unscaled:	 72.070 	l2 loss unscaled: 472.835
It 40	 Losses: total: 16.953,	target: -0.615 	R_feature_loss unscaled:	 54.542 	l2 loss unscaled: 460.104
It 50	 Losses: total: 13.936,	target: -0.622 	R_feature_loss unscaled:	 45.433 	l2 loss unscaled: 452.302
It 60	 Losses: total: 12.669,	target: -0.632 	R_feature_loss unscaled:	 41.647 	l2 loss unscaled: 449.736
It 70	 Losses: total: 11.418,	target: -0.644 	R_feature_loss unscaled:	 37.917 	l2 loss unscaled: 451.444
It 80	 Losses: total: 11.104,	target: -0.648 	R_feature_loss unscaled:	 36.987 	l2 loss unscaled: 454.365
It 90	 Losses: t